# Masks viewer

Pick an annotated image and see its masks. The masks come from the app's CSV export, the images from Girder.

1. In the app, open the project and press the download button at the bottom of the sidebar.
2. Start Jupyter from `backend/` with `GIRDER_TOKEN` set, or enter the key when asked:

```bash
UV_CACHE_DIR=/tmp/uv-cache uv run --with jupyterlab --with ipywidgets --with matplotlib --with pandas jupyter lab ../notebooks/masks.ipynb
```

After annotating more, download again and rerun from the next cell.

In [ ]:
import io
import os
from getpass import getpass
from pathlib import Path

import httpx
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shapely
from IPython.display import display
from matplotlib.patches import Patch
from PIL import Image, ImageDraw

# The newest export in Downloads; set a path to use another one.
CSV = None
# An image name or id to show first; None is the one annotated last.
IMAGE = None
MAX_SIDE = 2048
GIRDER_API_URL = os.environ.get("GIRDER_API_URL", "https://rhinodb.ims.bio/api/v1").rstrip("/")

In [ ]:
csv_path = (
    Path(CSV)
    if CSV
    else max(
        (Path.home() / "Downloads").glob("annotations-*.csv"),
        key=lambda p: p.stat().st_mtime,
        default=None,
    )
)
if csv_path is None:
    raise FileNotFoundError("No annotations-*.csv in Downloads: export one or set CSV.")
masks = pd.read_csv(csv_path, dtype={"image_id": str}).query("type == 'polygon'").copy()
if masks.empty:
    raise ValueError(f"{csv_path.name} contains no masks.")
masks["geometry"] = masks["geometry"].map(shapely.from_wkt)
masks["updated"] = pd.to_datetime(masks["updated"], utc=True)
images = (
    masks.groupby("image_id")
    .agg(image=("image", "first"), masks=("image", "size"), updated=("updated", "max"))
    .sort_values("updated", ascending=False)
)
jobs = sorted(masks["job"].unique())
colors = {job: plt.get_cmap("tab10")(i % 10)[:3] for i, job in enumerate(jobs)}
print(csv_path.name)
images

In [ ]:
def connect() -> httpx.Client:
    key = os.environ.get("GIRDER_TOKEN") or getpass("Girder API key: ")
    client = httpx.Client(base_url=GIRDER_API_URL, timeout=60, trust_env=False)
    me = client.get("/user/me", headers={"Girder-Token": key})
    token = key
    if not (me.status_code == 200 and me.json()):
        answer = client.post("/api_key/token", data={"key": key}).raise_for_status()
        token = answer.json()["authToken"]["token"]
    client.headers["Girder-Token"] = token
    return client


girder = connect()


def load(image_id: str) -> tuple[np.ndarray, tuple[int, int]]:
    """The image, at most MAX_SIDE on its long side, and its full (width, height)."""
    info = girder.get(f"/item/{image_id}/tiles").raise_for_status().json()
    full = info["sizeX"], info["sizeY"]
    size = {"width": min(MAX_SIDE, full[0]), "height": min(MAX_SIDE, full[1])}
    params = {**size, "encoding": "PNG"}
    png = girder.get(f"/item/{image_id}/tiles/thumbnail", params=params).raise_for_status()
    return np.asarray(Image.open(io.BytesIO(png.content)).convert("RGB")), full


def rasterize(geometries, full: tuple[int, int], shape: tuple[int, int]) -> np.ndarray:
    """Boolean mask of `shape` (height, width) for polygons in full-resolution pixels."""
    height, width = shape
    scale = np.array([width / full[0], height / full[1]])
    out = np.zeros(shape, dtype=bool)
    for geometry in geometries:
        for part in shapely.get_parts(geometry):
            if part.geom_type != "Polygon":
                continue
            canvas = Image.new("1", (width, height))
            draw = ImageDraw.Draw(canvas)
            for ring, fill in [(part.exterior, 1), *((hole, 0) for hole in part.interiors)]:
                draw.polygon([tuple(p) for p in shapely.get_coordinates(ring) * scale], fill=fill)
            out |= np.asarray(canvas)
    return out

In [ ]:
def show(image_id: str, shown: tuple[str, ...], opacity: float) -> None:
    pixels, full = load_cached(image_id)
    rows = masks[(masks["image_id"] == image_id) & masks["job"].isin(shown)]
    overlay = pixels / 255
    for job, group in rows.groupby("job"):
        mask = rasterize(group["geometry"], full, pixels.shape[:2])
        overlay[mask] = (1 - opacity) * overlay[mask] + opacity * np.array(colors[job])
    _, axes = plt.subplots(1, 2, figsize=(16, 8), sharex=True, sharey=True)
    axes[0].imshow(pixels)
    axes[1].imshow(overlay)
    handles = [
        Patch(color=colors[job], label=f"{job} ({n})")
        for job, n in rows["job"].value_counts().sort_index().items()
    ]
    if handles:
        axes[1].legend(handles=handles, loc="upper right")
    for ax in axes:
        ax.axis("off")
    plt.suptitle(images.loc[image_id, "image"])
    plt.tight_layout()
    plt.show()


_cache = {}


def load_cached(image_id: str) -> tuple[np.ndarray, tuple[int, int]]:
    if image_id not in _cache:
        _cache[image_id] = load(image_id)
    return _cache[image_id]


matches = [i for i, name in images["image"].items() if IMAGE in (i, name)]
if IMAGE and not matches:
    raise ValueError(f"{IMAGE!r} has no masks in {csv_path.name}.")

picker = widgets.Dropdown(
    options=[(f"{r.image} ({r.masks} masks)", i) for i, r in images.iterrows()],
    value=matches[0] if matches else images.index[0],
    description="Image",
    layout=widgets.Layout(width="500px"),
)
classes = widgets.SelectMultiple(options=jobs, value=jobs, description="Classes")
opacity = widgets.FloatSlider(0.45, min=0, max=1, step=0.05, description="Opacity")
controls = {"image_id": picker, "shown": classes, "opacity": opacity}
display(widgets.VBox([picker, classes, opacity]), widgets.interactive_output(show, controls))